In [1]:
# manejo de datos
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

# api Xenocanto
from xenopy import Query
from scipy import signal
import scipy.signal.windows

# manejo de audio
import librosa
import librosa.display

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
familia = os.listdir('songs/')[0]
os.path.basename(f'songs/{familia}')

def cargar_audios(ruta_base):
    audios_cargados = {}
    ruta_especie = os.path.join(ruta_base)
    if os.path.isdir(ruta_especie):
        especie = os.path.basename(ruta_especie)
        for nombre_comun in os.listdir(ruta_especie):
            ruta_nombre_comun = os.path.join(ruta_especie, nombre_comun)
            if os.path.isdir(ruta_nombre_comun):
                for archivo in os.listdir(ruta_nombre_comun):
                    if archivo.endswith('.mp3'):
                        ruta_archivo = os.path.join(ruta_nombre_comun, archivo)
                        try:
                            audio, sr = librosa.load(ruta_archivo)
                            # Construir el nombre clave como "especie_id.mp3"
                            nombre_clave = f"{especie}_{archivo.split('.')[0]}"
                            audios_cargados[nombre_clave] = (audio, sr)
                        except Exception as e:
                            print(f"Error al cargar {ruta_archivo}: {e}")
    return audios_cargados, especie

ruta_base = 'songs/Tyrannidae/Agriornis/Agriornis albicauda'
audios_cargados, especie = cargar_audios(ruta_base)
print(f"Se cargaron {len(audios_cargados)} archivos de audio.")

Se cargaron 20 archivos de audio.


In [3]:
def feature_extraction(audios_cargados, especie):
    all_features = [] 
    keys = list(audios_cargados.keys())
    for key in keys: 
        y, sr = audios_cargados[key]
        features = { 
            'Familia': familia,
            'Genero': especie.split(' ')[0],
            'Especie': especie,
            'id': key,
            'Longitud de onda': len(y),
            'Intensidad miníma': np.min(librosa.feature.rms(y=y)),
            'Intensidad media': np.mean(librosa.feature.rms(y=y)),
            'Intensidad máxima': np.max(librosa.feature.rms(y=y)),
            'Tonos principales': len(librosa.core.piptrack(y=y)[1]),
            'Melodía': np.mean(librosa.feature.tonnetz(y=y, sr=sr)),
            'Centroide espectral': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
            'Rolloff espectral': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)),
            'Contraste espectral': np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)),
            'Ancho de banda espectral': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)),
            'Croma': np.mean(librosa.feature.chroma_stft(y=y, sr=sr)),
            'Tempo': librosa.beat.tempo(y=y, sr=sr)[0],
            'Pulso': np.mean(librosa.feature.tempogram(y=y, sr=sr)),
            'Coeficientes MFCC': np.mean(librosa.feature.mfcc(y=y, sr=sr)),
            'RMS': np.mean(librosa.feature.rms(y=y)),
            'Cens': np.mean(librosa.feature.chroma_cens(y=y, sr=sr)),
            'Piptrack': np.mean(librosa.core.piptrack(y=y)[1]),
            'Cruces por cero': np.mean(librosa.feature.zero_crossing_rate(y)),
            'Cromagrama CQT': np.mean(librosa.feature.chroma_cqt(y=y, sr=sr)),
            'Cromagrama CENS': np.mean(librosa.feature.chroma_cens(y=y, sr=sr)),
            'Melspectrogram': np.mean(librosa.feature.melspectrogram(y=y, sr=sr)),
            'Polynomial coefficients': np.mean(librosa.feature.poly_features(y=y, sr=sr)),
            'Fourier tempogram': np.mean(librosa.feature.fourier_tempogram(y=y, sr=sr)),
            'Ceros Cruzados': np.mean(librosa.feature.zero_crossing_rate(y)),
            'Perceptrual Sharpness': np.mean(librosa.feature.spectral_flatness(y=y)),
            'Loudness': np.sum(librosa.feature.rms(y=y)),
            'Tonnetz': np.mean(librosa.feature.tonnetz(y=y, sr=sr)),
            'Contraste': np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)),
            'Rolloff': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)),
            'Centroide': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
            'Bandwidth': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
        }
        all_features.append(features) 
    df = pd.DataFrame(all_features)
    
    return df 

In [4]:
df = feature_extraction(audios_cargados, especie)
print(df.shape)
df

(20, 35)


,Familia,Genero,Especie,id,Longitud de onda,Intensidad miníma,Intensidad media,Intensidad máxima,Tonos principales,Melodía,Centroide espectral,Rolloff espectral,Contraste espectral,Ancho de banda espectral,Croma,Tempo,Pulso,Coeficientes MFCC,RMS,Cens,Piptrack,Cruces por cero,Cromagrama CQT,Cromagrama CENS,Melspectrogram,Polynomial coefficients,Fourier tempogram,Ceros Cruzados,Perceptrual Sharpness,Loudness,Tonnetz,Contraste,Rolloff,Centroide,Bandwidth
0,Alaudidae,Agriornis,Agriornis albicauda,Agriornis albicauda_115173,70848,0.000034,0.013576,0.093769,1025,-0.002856,4949.665067,8417.158695,20.751675,2716.525994,0.611250,129.199219,0.111866,-31.842852,0.013576,0.284967,0.035082,0.387439,0.636833,0.284967,0.037951,0.149657,0.276045-0.000206j,0.387439,0.121911,1.887075,-0.002856,20.751675,8417.158695,4949.665067,2716.525994
1,Alaudidae,Agriornis,Agriornis albicauda,Agriornis albicauda_115174,673920,0.001448,0.008888,0.182806,1025,0.004129,4100.346031,7398.919775,22.068119,2569.724456,0.590781,123.046875,0.268241,-30.044968,0.008888,0.283534,0.030966,0.277367,0.619477,0.283534,0.014941,0.123685,0.282404-0.001449j,0.277367,0.052673,11.705231,0.004129,22.068119,7398.919775,4100.346031,2569.724456
2,Alaudidae,Agriornis,Agriornis albicauda,Agriornis albicauda_163032,758592,0.000015,0.030219,0.423987,1025,-0.000450,2910.150791,4205.883542,21.204520,1721.282773,0.363812,129.199219,0.218826,-16.892817,0.030219,0.282370,0.046912,0.233121,0.540832,0.282370,0.254231,0.281609,0.688213-0.003314j,0.233121,0.014419,44.784252,-0.000450,21.204520,4205.883542,2910.150791,1721.282773
3,Alaudidae,Agriornis,Agriornis albicauda,Agriornis albicauda_163034,294336,0.000301,0.022171,0.457979,1025,0.000509,2857.283701,4660.290718,19.877762,2034.875373,0.526663,129.199219,0.182849,-26.073219,0.022171,0.286391,0.021907,0.204690,0.641446,0.286391,0.350665,0.178083,0.530612-0.025322j,0.204690,0.026561,12.748286,0.000509,19.877762,4660.290718,2857.283701,2034.875373
4,Alaudidae,Agriornis,Agriornis albicauda,Agriornis albicauda_250156,626112,0.000503,0.038549,0.231576,1025,-0.002837,3084.394749,6164.007044,19.619669,2639.087818,0.559212,123.046875,0.257183,-5.694504,0.038549,0.281872,0.115286,0.188409,0.655300,0.281872,0.286391,0.683951,0.529280-0.006453j,0.188409,0.085242,47.145767,-0.002837,19.619669,6164.007044,3084.394749,2639.087818
5,Alaudidae,Agriornis,Agriornis albicauda,Agriornis albicauda_332922,267840,0.000093,0.039316,0.199482,1025,0.002300,2404.868226,3374.076226,21.213392,1514.692552,0.458994,123.046875,0.208672,-16.392969,0.039316,0.284440,0.097000,0.174114,0.568394,0.284440,0.297109,0.513589,0.514905-0.011991j,0.174114,0.009984,20.601707,0.002300,21.213392,3374.076226,2404.868226,1514.692552
6,Alaudidae,Agriornis,Agriornis albicauda,Agriornis albicauda_364209,553536,0.000078,0.026013,0.297351,1025,0.005337,4392.166386,7638.864006,20.447755,2725.915257,0.604979,123.046875,0.242067,-22.577744,0.026013,0.283371,0.049117,0.332331,0.627483,0.283371,0.279735,0.264860,0.496961-0.005409j,0.332331,0.195227,28.146008,0.005337,20.447755,7638.864006,4392.166386,2725.915257
7,Alaudidae,Agriornis,Agriornis albicauda,Agriornis albicauda_364210,307584,0.000133,0.011922,0.086494,1025,0.011011,4570.478585,8468.191889,19.952340,3218.071933,0.630566,135.999178,0.214395,-11.770895,0.011922,0.280640,0.034541,0.350918,0.598590,0.280640,0.028230,0.171651,0.450372-0.010825j,0.350918,0.196830,7.165282,0.011011,19.952340,8468.191889,4570.478585,3218.071933
8,Alaudidae,Agriornis,Agriornis albicauda,Agriornis albicauda_501813,510336,0.000445,0.002299,0.015609,1025,0.001674,1960.581366,4511.821598,17.835550,2635.712107,0.642016,129.199219,0.277316,-16.722393,0.002299,0.283120,0.000502,0.071697,0.729825,0.283120,0.002673,0.020261,0.454000-0.003261j,0.071697,0.014105,2.292203,0.001674,17.835550,4511.821598,1960.581366,2635.712107
9,Alaudidae,Agriornis,Agriornis albicauda,Agriornis albicauda_501920,257472,0.000346,0.001476,0.016027,1025,0.004013,2299.132311,5112.616003,17.963029